# Lecture 10 Exercises

This notebook implements the GPU versions requested in Exercises 10.1 and 10.2.

Notes:
- The code assumes a CUDA-capable GPU.
- CuPy is optional; the notebook will use it only when it is installed.
- The benchmark cells are written so they can be run end-to-end once the GPU runtime is available.

In [7]:
from __future__ import annotations

import math
import time
from typing import Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numba import cuda, float32, int32
from numba.cuda.random import create_xoroshiro128p_states, xoroshiro128p_uniform_float32

try:
    import cupy as cp
    CUPY_AVAILABLE = True
except Exception:
    cp = None
    CUPY_AVAILABLE = False

plt.style.use('seaborn-v0_8-whitegrid')

CUDA_AVAILABLE = cuda.is_available()
print(f'CUDA available: {CUDA_AVAILABLE}')
print(f'CuPy available: {CUPY_AVAILABLE}')

PI_THREADS_PER_BLOCK = 256
PI_SAMPLES_PER_THREAD = 256
MATMUL_TILE_SIZES = (8, 16, 32)
MATMUL_BLOCK_DIM = 16


def require_cuda() -> None:
    if not CUDA_AVAILABLE:
        raise RuntimeError('A CUDA-capable GPU is required to run these kernels.')


def sync_if_cuda() -> None:
    if CUDA_AVAILABLE:
        cuda.synchronize()


def timed_call(fn, *args, **kwargs):
    sync_if_cuda()
    start = time.perf_counter()
    result = fn(*args, **kwargs)
    sync_if_cuda()
    elapsed = time.perf_counter() - start
    return result, elapsed

CUDA available: False
CuPy available: False


## Exercise 10.1 - Monte Carlo Estimation of pi

The first kernel below reproduces the Ex 9.3 baseline: each thread generates a stream of Monte Carlo samples, writes one boolean result per sample, and the host performs the final sum after copying the full result array back from the GPU.

The second kernel performs the final reduction on the GPU. Each thread generates several samples, accumulates a local count, and the block reduces those counts in shared memory. The final reduction stage can then be finished either on the host or by launching another reduction kernel on the GPU.

In [5]:
PI_BENCHMARK_SIZES = (10**6, 10**7, 10**8)
PI_HITS_DTYPE = np.uint8
PI_COUNT_DTYPE = np.int32


def pi_launch_config(
    n_samples: int,
    samples_per_thread: int = PI_SAMPLES_PER_THREAD,
    threads_per_block: int = PI_THREADS_PER_BLOCK,
) -> Tuple[int, int, int]:
    n_threads = math.ceil(n_samples / samples_per_thread)
    blocks = math.ceil(n_threads / threads_per_block)
    return blocks, threads_per_block, n_threads


@cuda.jit
def pi_hits_chunk_kernel(
    n_samples: int,
    samples_per_thread: int,
    n_threads: int,
    states,
    hits,
) -> None:
    tid = cuda.grid(1)
    if tid >= n_threads:
        return

    start = tid * samples_per_thread
    if start >= n_samples:
        return

    limit = samples_per_thread
    remaining = n_samples - start
    if remaining < limit:
        limit = remaining

    for offset in range(limit):
        x = xoroshiro128p_uniform_float32(states, tid)
        y = xoroshiro128p_uniform_float32(states, tid)
        hits[start + offset] = 1 if x * x + y * y <= 1.0 else 0


@cuda.jit
def pi_block_reduce_kernel(
    n_samples: int,
    samples_per_thread: int,
    n_threads: int,
    states,
    block_totals,
) -> None:
    shared = cuda.shared.array(PI_THREADS_PER_BLOCK, int32)
    tid = cuda.threadIdx.x
    gid = cuda.grid(1)
    local = 0

    if gid < n_threads:
        start = gid * samples_per_thread
        if start < n_samples:
            limit = samples_per_thread
            remaining = n_samples - start
            if remaining < limit:
                limit = remaining

            for _ in range(limit):
                x = xoroshiro128p_uniform_float32(states, gid)
                y = xoroshiro128p_uniform_float32(states, gid)
                if x * x + y * y <= 1.0:
                    local += 1

    shared[tid] = local
    cuda.syncthreads()

    step = cuda.blockDim.x // 2
    while step > 0:
        if tid < step:
            shared[tid] += shared[tid + step]
        cuda.syncthreads()
        step //= 2

    if tid == 0:
        block_totals[cuda.blockIdx.x] = shared[0]


@cuda.jit
def reduce_sum_kernel(values, partials) -> None:
    shared = cuda.shared.array(PI_THREADS_PER_BLOCK, int32)
    tid = cuda.threadIdx.x
    gid = cuda.grid(1)
    stride = cuda.gridsize(1)

    local = 0
    for idx in range(gid, values.size, stride):
        local += int(values[idx])

    shared[tid] = local
    cuda.syncthreads()

    step = cuda.blockDim.x // 2
    while step > 0:
        if tid < step:
            shared[tid] += shared[tid + step]
        cuda.syncthreads()
        step //= 2

    if tid == 0:
        partials[cuda.blockIdx.x] = shared[0]


# Keep a single GPU reduction helper rather than recursively copying intermediate values back.
def gpu_reduce_sum(device_values) -> int:
    require_cuda()
    current = device_values
    while current.size > 1:
        blocks = max(1, math.ceil(current.size / PI_THREADS_PER_BLOCK))
        d_next = cuda.device_array(blocks, dtype=PI_COUNT_DTYPE)
        reduce_sum_kernel[blocks, PI_THREADS_PER_BLOCK](current, d_next)
        cuda.synchronize()
        current = d_next
    return int(current.copy_to_host()[0])


def pi_from_hits(total_hits: int, n_samples: int) -> float:
    return 4.0 * total_hits / n_samples


def run_pi_transfer_everything(
    n_samples: int,
    seed: int = 0,
    samples_per_thread: int = PI_SAMPLES_PER_THREAD,
) -> Tuple[float, int]:
    require_cuda()
    blocks, threads_per_block, n_threads = pi_launch_config(n_samples, samples_per_thread)
    d_states = create_xoroshiro128p_states(n_threads, seed=seed)
    d_hits = cuda.device_array(n_samples, dtype=PI_HITS_DTYPE)
    pi_hits_chunk_kernel[blocks, threads_per_block](n_samples, samples_per_thread, n_threads, d_states, d_hits)
    cuda.synchronize()
    hits = d_hits.copy_to_host()
    return pi_from_hits(int(hits.sum()), n_samples), hits.nbytes


def run_pi_block_reduce_host_sum(
    n_samples: int,
    seed: int = 0,
    samples_per_thread: int = PI_SAMPLES_PER_THREAD,
) -> Tuple[float, int]:
    require_cuda()
    blocks, threads_per_block, n_threads = pi_launch_config(n_samples, samples_per_thread)
    d_states = create_xoroshiro128p_states(n_threads, seed=seed)
    d_block_totals = cuda.device_array(blocks, dtype=PI_COUNT_DTYPE)
    pi_block_reduce_kernel[blocks, threads_per_block](
        n_samples,
        samples_per_thread,
        n_threads,
        d_states,
        d_block_totals,
    )
    cuda.synchronize()
    block_totals = d_block_totals.copy_to_host()
    return pi_from_hits(int(block_totals.sum()), n_samples), block_totals.nbytes


def run_pi_block_reduce_gpu_sum(
    n_samples: int,
    seed: int = 0,
    samples_per_thread: int = PI_SAMPLES_PER_THREAD,
) -> Tuple[float, int]:
    require_cuda()
    blocks, threads_per_block, n_threads = pi_launch_config(n_samples, samples_per_thread)
    d_states = create_xoroshiro128p_states(n_threads, seed=seed)
    d_block_totals = cuda.device_array(blocks, dtype=PI_COUNT_DTYPE)
    pi_block_reduce_kernel[blocks, threads_per_block](
        n_samples,
        samples_per_thread,
        n_threads,
        d_states,
        d_block_totals,
    )
    cuda.synchronize()
    total_hits = gpu_reduce_sum(d_block_totals)
    return pi_from_hits(total_hits, n_samples), np.dtype(PI_COUNT_DTYPE).itemsize


if CUPY_AVAILABLE:
    def run_pi_cupy_sum(
        n_samples: int,
        seed: int = 0,
        chunk_size: int = 10_000_000,
    ) -> Tuple[float, int]:
        cp.random.seed(seed)
        remaining = n_samples
        total_hits = cp.array(0, dtype=cp.int64)

        while remaining > 0:
            current = min(chunk_size, remaining)
            x = cp.random.random(current).astype(cp.float32)
            y = cp.random.random(current).astype(cp.float32)
            total_hits += cp.sum((x * x + y * y) <= 1.0)
            remaining -= current

        return float(4.0 * total_hits.get() / n_samples), np.dtype(np.int64).itemsize
else:
    def run_pi_cupy_sum(*args, **kwargs):
        raise RuntimeError('CuPy is not available in this environment.')


def warm_up_pi_kernels() -> None:
    if not CUDA_AVAILABLE:
        return

    _ = run_pi_transfer_everything(1024, seed=0)
    _ = run_pi_block_reduce_host_sum(1024, seed=0)
    _ = run_pi_block_reduce_gpu_sum(1024, seed=0)

    if CUPY_AVAILABLE:
        _ = run_pi_cupy_sum(1024, seed=0, chunk_size=256)


PI_METHODS = [
    ('transfer_everything', run_pi_transfer_everything),
    ('block_reduce_host_sum', run_pi_block_reduce_host_sum),
    ('block_reduce_gpu_sum', run_pi_block_reduce_gpu_sum),
]
if CUPY_AVAILABLE:
    PI_METHODS.append(('cupy_sum', run_pi_cupy_sum))


def benchmark_pi_methods(
    sizes: Sequence[int] = PI_BENCHMARK_SIZES,
    seed: int = 0,
) -> pd.DataFrame:
    warm_up_pi_kernels()
    rows = []
    for n_samples in sizes:
        for method_name, method in PI_METHODS:
            (pi_estimate, gpu_to_cpu_bytes), elapsed = timed_call(method, n_samples, seed=seed)
            rows.append(
                {
                    'n_samples': n_samples,
                    'method': method_name,
                    'pi_estimate': pi_estimate,
                    'abs_error': abs(pi_estimate - math.pi),
                    'gpu_to_cpu_bytes': gpu_to_cpu_bytes,
                    'wall_time_s': elapsed,
                }
            )
    return pd.DataFrame(rows)


def plot_pi_benchmarks(results: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    for method_name, group in results.groupby('method'):
        group = group.sort_values('n_samples')
        ax.plot(group['n_samples'], group['wall_time_s'], marker='o', label=method_name)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('N samples')
    ax.set_ylabel('Wall time (s)')
    ax.set_title('Monte Carlo pi benchmark')
    ax.legend()
    fig.tight_layout()
    plt.show()


pi_results = None
# Uncomment the next line on a CUDA machine to run the benchmark table and plot.
# pi_results = benchmark_pi_methods()

if pi_results is not None:
    print(pi_results.to_string(index=False))
    plot_pi_benchmarks(pi_results)

## Exercise 10.2 - Matrix Multiplication on the GPU

The first kernel is the naive version: each thread computes one element of C by reading the corresponding row of A and column of B directly from global memory.

The second family of kernels uses shared-memory tiling. Each tile size is compiled into its own kernel so the notebook can compare TILE = 8, 16, and 32 cleanly.

In [6]:
MATMUL_BENCHMARK_SIZES = (256, 512, 1024, 2048)


def make_random_matrices(n: int, seed: int = 0) -> Tuple[np.ndarray, np.ndarray]:
    rng = np.random.default_rng(seed)
    a = rng.random((n, n), dtype=np.float32)
    b = rng.random((n, n), dtype=np.float32)
    return a.astype(np.float32, copy=False), b.astype(np.float32, copy=False)


@cuda.jit
def matmul_naive_kernel(a, b, c) -> None:
    row, col = cuda.grid(2)
    if row < c.shape[0] and col < c.shape[1]:
        acc = 0.0
        for k in range(a.shape[1]):
            acc += a[row, k] * b[k, col]
        c[row, col] = acc


def make_tiled_matmul_kernel(tile_size: int):
    @cuda.jit
    def tiled_matmul_kernel(a, b, c) -> None:
        tile_a = cuda.shared.array((tile_size, tile_size), dtype=float32)
        tile_b = cuda.shared.array((tile_size, tile_size), dtype=float32)
        row, col = cuda.grid(2)
        tx = cuda.threadIdx.x
        ty = cuda.threadIdx.y
        acc = 0.0
        num_tiles = (a.shape[1] + tile_size - 1) // tile_size

        for tile_idx in range(num_tiles):
            a_col = tile_idx * tile_size + tx
            b_row = tile_idx * tile_size + ty

            if row < a.shape[0] and a_col < a.shape[1]:
                tile_a[ty, tx] = a[row, a_col]
            else:
                tile_a[ty, tx] = 0.0

            if b_row < b.shape[0] and col < b.shape[1]:
                tile_b[ty, tx] = b[b_row, col]
            else:
                tile_b[ty, tx] = 0.0

            cuda.syncthreads()

            for k in range(tile_size):
                acc += tile_a[ty, k] * tile_b[k, tx]

            cuda.syncthreads()

        if row < c.shape[0] and col < c.shape[1]:
            c[row, col] = acc

    return tiled_matmul_kernel


TILED_MATMUL_KERNELS = {tile_size: make_tiled_matmul_kernel(tile_size) for tile_size in MATMUL_TILE_SIZES}


def gpu_matmul(a: np.ndarray, b: np.ndarray, kernel, block_dim: int) -> np.ndarray:
    require_cuda()
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    d_a = cuda.to_device(a)
    d_b = cuda.to_device(b)
    d_c = cuda.device_array((a.shape[0], b.shape[1]), dtype=np.float32)
    threads_per_block = (block_dim, block_dim)
    blocks_per_grid = (
        math.ceil(b.shape[1] / block_dim),
        math.ceil(a.shape[0] / block_dim),
    )
    kernel[blocks_per_grid, threads_per_block](d_a, d_b, d_c)
    cuda.synchronize()
    return d_c.copy_to_host()


if CUPY_AVAILABLE:
    def cupy_matmul(a: np.ndarray, b: np.ndarray) -> np.ndarray:
        d_a = cp.asarray(np.asarray(a, dtype=np.float32))
        d_b = cp.asarray(np.asarray(b, dtype=np.float32))
        d_c = cp.matmul(d_a, d_b)
        return d_c.get()
else:
    def cupy_matmul(*args, **kwargs):
        raise RuntimeError('CuPy is not available in this environment.')


MATMUL_METHODS = [
    ('naive', matmul_naive_kernel, MATMUL_BLOCK_DIM),
]
for tile_size in MATMUL_TILE_SIZES:
    MATMUL_METHODS.append((f'tiled_{tile_size}', TILED_MATMUL_KERNELS[tile_size], tile_size))

if CUPY_AVAILABLE:
    MATMUL_METHODS.append(('cupy_matmul', None, None))


def warm_up_matmul_kernels() -> None:
    if not CUDA_AVAILABLE:
        return

    a, b = make_random_matrices(32, seed=0)
    _ = gpu_matmul(a, b, matmul_naive_kernel, MATMUL_BLOCK_DIM)
    for tile_size in MATMUL_TILE_SIZES:
        _ = gpu_matmul(a, b, TILED_MATMUL_KERNELS[tile_size], tile_size)

    if CUPY_AVAILABLE:
        _ = cupy_matmul(a, b)


def assert_matmul_correctness() -> None:
    require_cuda()
    test_sizes = (4, 7, 16, 31, 64)

    for n in test_sizes:
        a, b = make_random_matrices(n, seed=n)
        expected = a @ b

        naive = gpu_matmul(a, b, matmul_naive_kernel, MATMUL_BLOCK_DIM)
        np.testing.assert_allclose(naive, expected, rtol=1e-4, atol=1e-4)

        for tile_size in MATMUL_TILE_SIZES:
            tiled = gpu_matmul(a, b, TILED_MATMUL_KERNELS[tile_size], tile_size)
            np.testing.assert_allclose(tiled, expected, rtol=1e-4, atol=1e-4)

    print('All matmul correctness tests passed.')


def benchmark_matmul_methods(
    sizes: Sequence[int] = MATMUL_BENCHMARK_SIZES,
) -> pd.DataFrame:
    require_cuda()
    warm_up_matmul_kernels()
    rows = []

    for n in sizes:
        a, b = make_random_matrices(n, seed=n)
        expected_output_bytes = n * n * np.dtype(np.float32).itemsize

        for method_name, kernel, block_dim in MATMUL_METHODS:
            if method_name == 'cupy_matmul':
                c_result, elapsed = timed_call(cupy_matmul, a, b)
            else:
                c_result, elapsed = timed_call(gpu_matmul, a, b, kernel, block_dim)

            rows.append(
                {
                    'n': n,
                    'method': method_name,
                    'wall_time_s': elapsed,
                    'gpu_to_cpu_bytes': c_result.nbytes if hasattr(c_result, 'nbytes') else expected_output_bytes,
                }
            )

    return pd.DataFrame(rows)


def plot_matmul_benchmarks(results: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    for method_name, group in results.groupby('method'):
        group = group.sort_values('n')
        ax.plot(group['n'], group['wall_time_s'], marker='o', label=method_name)

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Matrix size N')
    ax.set_ylabel('Wall time (s)')
    ax.set_title('Matrix multiplication benchmark')
    ax.legend()
    fig.tight_layout()
    plt.show()


# Correctness tests are defined above and can be run on a CUDA machine.
# assert_matmul_correctness()

matmul_results = None
# Uncomment the next line on a CUDA machine to run the benchmark table and plot.
# matmul_results = benchmark_matmul_methods()

if matmul_results is not None:
    print(matmul_results.to_string(index=False))
    plot_matmul_benchmarks(matmul_results)

## What to look for in the results

For Exercise 10.1, the main trend should be that the amount of data copied back to the host drops sharply as the reduction moves deeper onto the GPU. The transfer-everything baseline moves $O(N)$ values from GPU to CPU, the block-reduce + host-sum version moves only one partial per block, and the second GPU reduction version moves only a single scalar back to the host.

For Exercise 10.2, tiling reduces global-memory traffic because each element of A and B is reused by many threads inside a block. In the naive kernel, each output element reads an entire row and column from global memory. In the tiled kernel, each tile of A and B is loaded once per tile step and reused from shared memory, so the global-memory reads per element drop by roughly a factor of TILE.

If CuPy is available, its `cp.sum` and `cp.matmul` baselines are useful reference points for how much a hand-written Numba kernel still leaves on the table compared with library-optimized CUDA code.